In [11]:
import numpy as np
import pandas as pd

# --- Duka Electronics Generator ---
# Note: I am assuming the Duka generator looks like this based on standard PLP assignments.
# If your portal has a specific code block for Duka, replace this section with it.
np.random.seed(2026)
duka_data = pd.DataFrame({
    'order_id': [f"ORD-{i:04d}" for i in range(1, 1001)],
    'date': pd.date_range("2026-01-01", periods=1000, freq="6H").strftime("%Y-%m-%d"),
    'branch': np.random.choice(["Nairobi", "Mombasa", "Kisumu"], 1000),
    'channel': np.random.choice(["Online", "In-Store"], 1000),
    'product': np.random.choice(["Laptop", "Phone", "TV", "Headphones", "Charger"], 1000),
    'units': np.random.randint(1, 5, 1000),
    'price': np.random.randint(500, 30000, 1000)
})
duka_data['revenue'] = duka_data['units'] * duka_data['price']
duka_data.to_csv("duka_sales.csv", index=False)

# --- Agent Network Generator ---
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d")
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d")
})

idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

/tmp/ipykernel_617/3630307437.py:10: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'date': pd.date_range("2026-01-01", periods=1000, freq="6H").strftime("%Y-%m-%d"),


#Engagement 1: Duka Electronics
# Task 1:Data loading and First-Five-Minutes Inspection

**Inspection Notes**
**Shape** The data has 1000 rows and 8 columns.
**Types** The 'date' column is an object and needs to be pursed as a datetime. 'price' and 'units' are integers.
**Missing Values** There are no missing values in the dataset.
**Key Categories**Branches are Nairobi,Mombasa,Kisumu. Channels are online and In-store.


In [12]:
import pandas as pd
import numpy as np

# 1. Load the data and parse dates
df_duka = pd.read_csv("duka_sales.csv", parse_dates=['date'])

# 2. Print the inspection
print("Shape:", df_duka.shape)
print("\nData Types:\n", df_duka.dtypes)
print("\nMissing Values:\n", df_duka.isna().sum())
print("\nDate Range:", df_duka['date'].min(), "to", df_duka['date'].max())
print("\nBranches:", df_duka['branch'].unique())
print("\nChannels:", df_duka['channel'].unique())
print("\nFirst 5 rows:\n", df_duka.head())

Shape: (1000, 8)

Data Types:
 order_id            object
date        datetime64[ns]
branch              object
channel             object
product             object
units                int64
price                int64
revenue              int64
dtype: object

Missing Values:
 order_id    0
date        0
branch      0
channel     0
product     0
units       0
price       0
revenue     0
dtype: int64

Date Range: 2026-01-01 00:00:00 to 2026-09-07 00:00:00

Branches: ['Mombasa' 'Kisumu' 'Nairobi']

Channels: ['In-Store' 'Online']

First 5 rows:
    order_id       date   branch   channel     product  units  price  revenue
0  ORD-0001 2026-01-01  Mombasa  In-Store     Charger      2  28178    56356
1  ORD-0002 2026-01-01   Kisumu  In-Store      Laptop      2   6296    12592
2  ORD-0003 2026-01-01   Kisumu  In-Store       Phone      2  12162    24324
3  ORD-0004 2026-01-01  Nairobi  In-Store  Headphones      1  18418    18418
4  ORD-0005 2026-01-02  Mombasa  In-Store  Headphones      4  21

##Task 2:Derived Columns
*My approach: I will extract the monthand weekday name from the date.I'll create a boolean for weekends,and a price_band column where revenue is 28000 or more.*

In [13]:
# Month (e.g., 'January')
df_duka['month'] = df_duka['date'].dt.month_name()

# Weekday (e.g., 'Monday')
df_duka['weekday'] = df_duka['date'].dt.day_name()

# is_weekend (True if Saturday or Sunday)
df_duka['is_weekend'] = df_duka['date'].dt.dayofweek >= 5

# price_band (Premium if revenue >= 28000, else Value)
df_duka['price_band'] = np.where(df_duka['revenue'] >= 28000, 'Premium', 'Value')

print(df_duka[['date', 'month', 'weekday', 'is_weekend', 'revenue', 'price_band']].head())

        date    month   weekday  is_weekend  revenue price_band
0 2026-01-01  January  Thursday       False    56356    Premium
1 2026-01-01  January  Thursday       False    12592      Value
2 2026-01-01  January  Thursday       False    24324      Value
3 2026-01-01  January  Thursday       False    18418      Value
4 2026-01-02  January    Friday       False    86252    Premium


##Task 3: Rank Branches
*My approach:I will use 'groupby' to sum revenue and count orders per branch.*

In [14]:
# Rank by Revenue
branch_revenue = df_duka.groupby('branch')['revenue'].sum().sort_values(ascending=False)
print("Rank by Revenue:\n", branch_revenue)

# Rank by Number of Orders
branch_orders = df_duka.groupby('branch')['order_id'].count().sort_values(ascending=False)
print("\nRank by Number of Orders:\n", branch_orders)

Rank by Revenue:
 branch
Mombasa    12990627
Nairobi    12725957
Kisumu     12231205
Name: revenue, dtype: int64

Rank by Number of Orders:
 branch
Mombasa    340
Nairobi    331
Kisumu     329
Name: order_id, dtype: int64


*The rankings are [same/different].If they differ,it is because a branch might have fewer orders but sell much higher-value items(like Tvs/laptops),leading to higher revenue despite lower volume.*

## Task 4:Top products by Revenue and Units
*My approach:I will use group by product and sum both revenue and units.*

In [15]:
# Top by Revenue
top_rev = df_duka.groupby('product')['revenue'].sum().sort_values(ascending=False).head(3)
print("Top 3 Products by Revenue:\n", top_rev)

# Top by Units
top_units = df_duka.groupby('product')['units'].sum().sort_values(ascending=False).head(3)
print("\nTop 3 Products by Units Sold:\n", top_units)

Top 3 Products by Revenue:
 product
TV            8291880
Laptop        8155229
Headphones    7710187
Name: revenue, dtype: int64

Top 3 Products by Units Sold:
 product
Laptop    543
TV        518
Phone     492
Name: units, dtype: int64


## Task 5:weekend vs weekday orders
* My approach:I will use group by the 'is_weekend' boolean to compare the metrics.*

In [16]:
weekend_stats = df_duka.groupby('is_weekend').agg(
    number_of_orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_revenue=('revenue', 'mean')
)
print(weekend_stats)

            number_of_orders  total_revenue  avg_order_revenue
is_weekend                                                    
False                    712       27233698       38249.575843
True                     288       10714091       37201.704861


## Task 6:Best and Worst Month
*My approach: Group by month,sum revenue,and find the max/min.*

In [17]:
monthly_rev = df_duka.groupby('month')['revenue'].sum().sort_values(ascending=False)

best_month = monthly_rev.index[0]
best_rev = monthly_rev.iloc[0]
worst_month = monthly_rev.index[-1]
worst_rev = monthly_rev.iloc[-1]

pct_diff = ((best_rev - worst_rev) / worst_rev) * 100

print(f"Best Month: {best_month} (KES {best_rev:,})")
print(f"Worst Month: {worst_month} (KES {worst_rev:,})")
print(f"Percentage Difference: {pct_diff:.2f}%")

Best Month: July (KES 5,002,499)
Worst Month: September (KES 1,191,621)
Percentage Difference: 319.81%


## Task 7:Filter Premium Online Coast Capital Orders
*My approach: Filter for price_band=='premium',channel=='online',and branch in ['Nairobu','Mombasa'].Then export to csv.*

In [18]:
# Create the filter mask
filter_mask = (
    (df_duka['price_band'] == 'Premium') &
    (df_duka['channel'] == 'Online') &
    (df_duka['branch'].isin(['Nairobi', 'Mombasa']))
)

premium_online_coast = df_duka[filter_mask]

print(f"Number of orders: {len(premium_online_coast)}")
print(f"Total Revenue: KES {premium_online_coast['revenue'].sum():,}")

# Export to CSV (saved in Colab files)
premium_online_coast.to_csv("premium_online_coast_capital.csv", index=False)

Number of orders: 168
Total Revenue: KES 10,006,895


## Task 9: Executive Summary
- Duka Electronics generated a total revenue of KES [Insert number from Task 3] across 1,000 orders.
- The Mombasa branch was the top performer, generating KES [12990627] in revenue.
- Tvs were the highest revenue-generating product, bringing in KES [8291880].
- Weekend orders have an average revenue of KES [38249.57], compared to KES [37201.70] on weekdays.
- The best performing month was [July] with KES [5,002,499] in revenue, which was [319.81%] higher than the worst month.